In [7]:
import os
from PIL import Image
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models
from collections import Counter
from PIL import ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True
from tensorflow.keras.applications import VGG16, ResNet50, MobileNetV2
from tensorflow.keras.layers import Dense, Flatten, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.applications.vgg16 import preprocess_input as vgg_pre
from tensorflow.keras.layers import Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.applications.resnet50 import preprocess_input as res_pre
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mob_pre
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import numpy as np

In [2]:
train_dir = "Dataset/train"
val_dir = "Dataset/val"

In [3]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    horizontal_flip=True,
    zoom_range=0.2
)
val_datagen = ImageDataGenerator(rescale=1./255)

In [10]:
train_ds = train_datagen.flow_from_directory(
    train_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode="binary"      
)
val_ds = val_datagen.flow_from_directory(
    val_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode="binary"      
)
print("Số lớp:", train_ds.num_classes)


Found 14164 images belonging to 2 classes.
Found 1201 images belonging to 2 classes.
Số lớp: 2


In [5]:
base_model_resnet = ResNet50(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

for layer in base_model_resnet.layers:
    layer.trainable = False

In [ ]:
x = base_model_resnet.output
x = BatchNormalization()(x) 
x = Flatten()(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.4)(x)
output = Dense(1, activation='sigmoid')(x) 
model_resnet = Model(inputs=base_model_resnet.input, outputs=output)

In [ ]:

model_resnet.compile(
    optimizer='adam',
    loss='binary_crossentropy', 
    metrics=['accuracy']
)


In [11]:
history_resnet = model_resnet.fit(
    train_ds,            
    validation_data=val_ds, 
    epochs=3,            
)

c:\Users\vhgam\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/3
443/443 ━━━━━━━━━━━━━━━━━━━━ 965s 2s/step - accuracy: 0.8877 - loss: 0.7378 - val_accuracy: 0.7419 - val_loss: 0.5808
Epoch 2/3
443/443 ━━━━━━━━━━━━━━━━━━━━ 962s 2s/step - accuracy: 0.8965 - loss: 0.2953 - val_accuracy: 0.7435 - val_loss: 0.6927
Epoch 3/3
443/443 ━━━━━━━━━━━━━━━━━━━━ 994s 2s/step - accuracy: 0.8990 - loss: 0.2720 - val_accuracy: 0.7660 - val_loss: 0.7499


In [13]:
# lưu mô hình
model_resnet.save('resnet50_model.h5')

In [12]:
# đánh giá mô hình tren tập dữ liệu validation
val_loss, val_accuracy = model_resnet.evaluate(val_ds)
print(f'Validation Loss: {val_loss}')
print(f'Validation Accuracy: {val_accuracy}')

38/38 ━━━━━━━━━━━━━━━━━━━━ 59s 2s/step - accuracy: 0.7660 - loss: 0.7499
Validation Loss: 0.7499184608459473
Validation Accuracy: 0.7660282850265503
